<center>
    <p style="text-align:center">
    <img alt="arize logo" src="https://storage.googleapis.com/arize-assets/arize-logo-white.jpg" width="300"/>
        <br>
        <a href="https://docs.arize.com/arize/">Docs</a>
        |
        <a href="https://github.com/Arize-ai/client_python">GitHub</a>
        |
        <a href="https://arize-ai.slack.com/join/shared_invite/zt-11t1vbu4x-xkBIHmOREQnYnYDH1GDfCg">Community</a>
    </p>
</center>

# <center>OpenAI Realtime API Voice Tracing</center>

This notebook shows how to trace an OpenAI Realtime voice agent end-to-end with Arize AX, using the [OpenAI Agents SDK](https://github.com/openai/openai-agents-python) and [`openinference-instrumentation-openai-agents`](https://pypi.org/project/openinference-instrumentation-openai-agents/). You will learn:

- How a single `OpenAIAgentsInstrumentor().instrument(...)` call auto-traces an `agents.realtime.RealtimeSession` — capturing audio, transcripts, token counts, and tool calls without any manual span code
- The span tree the instrumentor produces (one `AUDIO` parent per conversational turn, with `USER`, `LLM`, and `TOOL` children)
- Two ways to store the captured audio — inline base64 data URIs (zero infrastructure) or Google Cloud Storage uploads (durable, arbitrary length)
- The Agents-SDK event stream — `RealtimeAudio`, `RealtimeAudioInterrupted`, `RealtimeAgentEndEvent`, `RealtimeToolStart` — and how to drive a live microphone + speaker loop on top of it

Each section runs a small piece of code, then guides you through what to look for in the Arize AX trace view.

### **NOTE**: This notebook must be run locally

Audio capture requires microphone access, which Colab cannot provide.

## What you'll see in Arize

The OpenAI Agents instrumentor turns each turn of a Realtime conversation into a complete span tree, automatically. You write no tracing code yourself — the instrumentor patches `RealtimeSession` and emits spans as audio, tool calls, and responses flow through the WebSocket.

For each conversational turn, the tree looks like this:

```
AUDIO  "conversation.turn"     ← parent; aggregated transcripts, llm.model_name, llm.invocation_parameters
├─ USER "user"                 ← input.audio.url, input.audio.transcript
├─ LLM  "assistant"            ← output.audio.url, output.audio.transcript, token counts, time_to_first_token_ms
│  └─ TOOL "<tool_name>"        ← one per function call within the turn
└─ ...                          ← additional siblings for split input or tool round-trips
```

A few details worth knowing:

- The parent span uses the `AUDIO` [OpenInference span kind](https://github.com/Arize-ai/openinference/tree/main/spec) — Arize AX renders it with audio-aware UI: a play button, waveform, and the input/output transcripts inline.
- `input.audio.url` and `output.audio.url` carry whatever URL exposes the audio. Variant 1 below uses a `data:audio/wav;base64,…` URI (the bytes ride on the span itself). Variant 2 uses an `https://storage.cloud.google.com/…` URL pointing at a GCS object the wrapping exporter uploaded.
- `time_to_first_token_ms` on the assistant span is the latency from the user's audio commit to the first byte of the model's audio response — the metric that matters most for perceived voice-agent responsiveness.
- Token counts include the audio-specific breakdowns `llm.token_count.prompt_details.audio` and `llm.token_count.completion_details.audio`, so you can see audio vs text token cost separately.

## Initial setup

You will need an Arize AX account to run this notebook. [Sign up now for free](https://app.arize.com/auth/join) if you don't have one. You also need an OpenAI API key with access to the Realtime API.

### Install libraries

The cell below installs the OpenAI Agents SDK, the OpenInference instrumentor, the `arize-otel` helper for tracer setup, the OTLP gRPC exporter (used by Variant 2), `google-cloud-storage` (also Variant 2), and `sounddevice` + `numpy` for audio I/O. On Linux it also installs the PortAudio system library that `sounddevice` links against; macOS and Windows wheels bundle PortAudio already.

In [ ]:
import shutil
import sys

if 'google.colab' in sys.modules:
    print("THIS NOTEBOOK MUST RUN LOCALLY. Colab environments do not support microphone capture.")
else:
    # sounddevice's prebuilt wheels bundle PortAudio on macOS and Windows.
    # On Linux you need the system library — install it via the available package manager.
    if sys.platform == "darwin" and shutil.which("brew"):
        !brew list portaudio >/dev/null 2>&1 || brew install portaudio
    elif sys.platform.startswith("linux"):
        if shutil.which("apt-get"):
            !sudo apt-get update && sudo apt-get install -y libportaudio2
        elif shutil.which("dnf"):
            !sudo dnf install -y portaudio
        elif shutil.which("pacman"):
            !sudo pacman -S --noconfirm portaudio
        else:
            print("Install the PortAudio system library before continuing (e.g. apt-get install libportaudio2).")
    # Windows: nothing to do — PortAudio ships in the sounddevice wheel.
    %pip install -q "openai-agents" "openinference-instrumentation-openai-agents" "arize-otel" "opentelemetry-exporter-otlp-proto-grpc" "google-cloud-storage" "sounddevice" "numpy"

### Imports

We import the standard library pieces (`asyncio`, `base64`, `os`, `subprocess`, `datetime`, `getpass`), the audio stack (`numpy` and `sounddevice`), and the Agents SDK objects that build the realtime agent. From `agents.realtime.events` we pull the specific event types we'll dispatch on inside the run loop. Finally, `OpenAIAgentsInstrumentor` is the one-line auto-instrumentor that wires the Agents SDK to OpenTelemetry.

In [ ]:
import asyncio
import base64
import os
import subprocess
from datetime import datetime
from getpass import getpass

import numpy as np
import sounddevice as sd

from agents import function_tool
from agents.realtime import RealtimeAgent, RealtimeRunner
from agents.realtime.events import (
    RealtimeAgentEndEvent,
    RealtimeAgentStartEvent,
    RealtimeAudio,
    RealtimeAudioInterrupted,
    RealtimeToolStart,
)

from openinference.instrumentation.openai_agents import OpenAIAgentsInstrumentor

### Set up credentials

Enter your OpenAI API key and Arize Space ID + API key. These prompts skip themselves if the env vars are already set.

You can find your Arize Space ID and API key in your space settings: [Arize AX → Space Settings](https://app.arize.com/). `PROJECT_NAME` is the name that will appear in the Arize project picker — change it if you want to group runs separately.

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")

if not os.environ.get("ARIZE_SPACE_ID"):
    os.environ["ARIZE_SPACE_ID"] = getpass("Enter ARIZE_SPACE_ID: ")

if not os.environ.get("ARIZE_API_KEY"):
    os.environ["ARIZE_API_KEY"] = getpass("Enter ARIZE_API_KEY: ")

PROJECT_NAME = "openai-realtime-voice"

## Defining the voice agent

Three pieces make up the agent we'll wire to the Realtime API: the **tools** (Python functions the model can call), the **agent definition** (instructions and a tool list), and the **audio plumbing** that pipes the microphone in and the speaker out. None of this code is tracing-specific — it's the same shape you'd write without observability. The instrumentor we configure in the next section will trace it all automatically.

### Tools

Two tools: `get_weather` and `get_current_time`. They're trivial dummies (the weather doesn't really come from anywhere), enough to exercise a tool round-trip in the Realtime API.

The `@function_tool` decorator from the Agents SDK introspects the function's type hints and docstring to generate the JSON schema the model receives. You don't write a tool-spec dict by hand. When the model decides to call `get_weather`, the SDK deserialises the arguments, invokes the Python function, serialises the result, and sends it back to the model — all of which the OpenInference instrumentor captures as a child `TOOL` span on the trace.

In [ ]:
@function_tool
def get_weather(location: str, unit: str = "fahrenheit") -> str:
    """Get the current weather for a location."""
    if unit == "celsius":
        return f"The weather in {location} is 22 °C and sunny."
    return f"The weather in {location} is 72 °F and sunny."


@function_tool
def get_current_time(timezone: str = "UTC") -> str:
    """Get the current time for a timezone."""
    now = datetime.now().strftime("%I:%M %p")
    return f"The current time in {timezone} is {now}."

### The agent

A `RealtimeAgent` is the Agents-SDK object that pairs a system prompt with a tool list for the OpenAI Realtime API. It looks identical to a regular `Agent` but is consumed by `RealtimeRunner` instead of `Runner`, which negotiates the WebSocket connection and handles the bidirectional audio stream.

The `instructions` you pass become the session's system message — kept short and explicit about when to call tools and how to reply, since long instructions slow down the time-to-first-token at the start of every turn. You can pass `model="..."` if you want a non-default Realtime model; left unset, the SDK uses its current default.

In [ ]:
agent = RealtimeAgent(
    name="Assistant",
    instructions=(
        "You are a helpful voice assistant. "
        "You have tools to look up weather and the current time — use them when asked. "
        "Keep responses short and conversational."
    ),
    tools=[get_weather, get_current_time],
)

### Audio plumbing

The OpenAI Realtime API speaks 24 kHz PCM16 mono on the wire — that's what `session.send_audio(...)` expects and what `RealtimeAudio` events carry on the way back. We use [`sounddevice`](https://python-sounddevice.readthedocs.io/) to bridge the system's microphone and speaker to that format.

`sounddevice` runs the input stream's callback on a background thread; the callback enqueues mic bytes into an `asyncio.Queue` that the async send loop drains. For playback we use a non-callback `OutputStream` and write to it synchronously via `loop.run_in_executor(...)`, letting PortAudio own the buffering. When the assistant's reply ends, we call `out_stream.stop()`, which blocks until PortAudio has played out the queued audio — so we never need a manual drain loop.

In [ ]:
def make_mic_callback(mic_queue, loop):
    """Build a sounddevice InputStream callback that forwards mic chunks into an asyncio queue.

    sounddevice invokes the callback on PortAudio's background thread, so we use
    `loop.call_soon_threadsafe` to hand the bytes to the asyncio event loop safely.
    """
    def cb(indata, frames, _time, status):
        if status:
            # Underruns, overflows etc. — print but don't crash; PortAudio recovers.
            print(f"Mic: {status}")
        # `indata` is a view onto a buffer sounddevice reuses on the next callback,
        # so we copy before handing the bytes to a different thread.
        loop.call_soon_threadsafe(mic_queue.put_nowait, indata.copy().tobytes())
    return cb

## Setting up tracing

This is the only tracing code you need to write for the whole notebook — one `register(...)` call and one `OpenAIAgentsInstrumentor().instrument(...)` call. The instrumentor patches `agents.realtime.RealtimeSession` to emit the span tree shown above; all spans flow through the registered `TracerProvider` to Arize AX.

The notebook offers two variants for where the captured audio bytes live:

| | Variant 1 — Embedded base64 | Variant 2 — Google Cloud Storage |
|---|---|---|
| Where audio lives | Inline on the span attribute as `data:audio/wav;base64,…` | A GCS object; span attribute holds the `https://storage.cloud.google.com/…` URL |
| Setup cost | None — works out of the box | Requires `gcloud` auth + an existing bucket |
| Length limit | Capped by `OPENINFERENCE_BASE64_AUDIO_MAX_LENGTH` (default 32 000 chars ≈ 0.5 s; we raise it below) | No practical limit — bound by the GCS object size limit |
| Best for | Tutorials, demos, short turns | Production workloads, long turns, durable storage |

**Run exactly one of the two variant cells below — not both.** The instrumentor patches global SDK state on `.instrument(...)`, so if you want to switch variants, restart the kernel first.

### Variant 1 — Embedded base64 audio

The simplest path. The instrumentor encodes each audio buffer as a `data:audio/wav;base64,…` URI and sets it directly on the `user` / `assistant` spans. No cloud account, no upload step, no auth. Arize AX plays back the data URI directly in the trace view.

`arize.otel.register(...)` returns a configured `TracerProvider` with an OTLP exporter that ships traces to Arize using your space ID and API key as authentication headers. We then call `OpenAIAgentsInstrumentor().instrument(tracer_provider=...)` to patch the Agents SDK.

The base64 cap is controlled by the `OPENINFERENCE_BASE64_AUDIO_MAX_LENGTH` env var. The default (`32000` characters) is roughly 0.5 s of 24 kHz mono PCM16; we bump it to `2000000` (~30 s) here. **Set this env var before calling `instrument(...)`** — the instrumentor reads it at patch time.

In [ ]:
from arize.otel import register

# Raise the inline base64 cap to ~30 s of audio (default is ~0.5 s).
# Must be set BEFORE .instrument() — the instrumentor reads it at patch time.
os.environ["OPENINFERENCE_BASE64_AUDIO_MAX_LENGTH"] = "2000000"

# arize.otel.register builds a TracerProvider with an OTLP exporter wired to Arize AX,
# and installs it as the global OpenTelemetry tracer provider.
tracer_provider = register(
    space_id=os.environ["ARIZE_SPACE_ID"],
    api_key=os.environ["ARIZE_API_KEY"],
    project_name=PROJECT_NAME,
)

# Patch agents.realtime.RealtimeSession so it emits spans through our tracer provider.
OpenAIAgentsInstrumentor().instrument(tracer_provider=tracer_provider)

### Variant 2 — Upload audio to Google Cloud Storage

Use this when you need durable storage, support for long turns (the embedded base64 path is capped per-span), or you already have a GCS bucket as part of your infrastructure. The pattern is a thin wrapping `SpanExporter` that sits between the OpenInference-tagged span and the OTLP exporter shipping to Arize.

The exporter does three things on each span:

1. Inspects `input.audio.url` / `output.audio.url` for `data:audio/...` URIs (the inline base64 the instrumentor would otherwise ship to Arize)
2. Decodes the base64, uploads the WAV bytes to GCS at a path keyed by trace + span ID
3. Reconstructs the span with the GCS URL substituted for the data URI, and forwards it to the real OTLP exporter

The inline base64 never reaches Arize — only the short GCS URL does. We raise `OPENINFERENCE_BASE64_AUDIO_MAX_LENGTH` to effectively unlimited so the full audio is available for upload, and we set up the `TracerProvider` manually here (rather than via `arize.otel.register`) so we can control the exporter chain.

First, fill in the bucket and (optional) path prefix, then authenticate with `gcloud`:

In [ ]:
# Fill these in for the GCS variant.
GCS_BUCKET = ""  # e.g. "my-arize-audio"
GCS_PATH = ""    # e.g. "voice-traces/" (trailing slash is added automatically)


def check_bucket_access(bucket_name):
    subprocess.run(
        ["gsutil", "ls", f"gs://{bucket_name}"],
        check=True,
        capture_output=True,
    )
    print(f"Access to bucket '{bucket_name}' verified.")


if GCS_BUCKET:
    try:
        subprocess.run(["gcloud", "auth", "application-default", "login"], check=True)
        check_bucket_access(GCS_BUCKET)
    except subprocess.CalledProcessError as e:
        print(f"Auth or bucket access failed: {e}")
else:
    print("GCS_BUCKET is empty — set it before running the next cell.")

Now we define the wrapping exporter and wire up the `TracerProvider` manually. `ReadableSpan` attributes are immutable once a span has ended, so rewriting the audio URL means **constructing a new `ReadableSpan`** with the substituted attribute (copying every other field through unchanged) and passing that to the inner OTLP exporter.

The file path inside the bucket is `<timestamp>_<trace_id>_<span_id>_<input|output>.wav` (with the optional path prefix prepended) — keyed by trace and span ID so each audio clip is uniquely addressable from its span.

In [ ]:
from google.cloud import storage
from opentelemetry import trace as trace_api
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import ReadableSpan, TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, SpanExporter

# Span attributes the instrumentor sets the inline audio data URI on.
AUDIO_URL_KEYS = ("input.audio.url", "output.audio.url")


class GCSAudioUploadingExporter(SpanExporter):
    """Wraps an OTLP exporter; for each span carrying a `data:audio/...` URL,
    uploads the decoded audio to GCS and substitutes the GCS URL before
    handing the span off to the real exporter."""

    def __init__(self, wrapped, bucket_name, path_prefix=""):
        self._wrapped = wrapped
        self._bucket = storage.Client().bucket(bucket_name)
        self._prefix = path_prefix.strip("/")

    # SpanExporter interface — delegate to the wrapped OTLP exporter, with each
    # span rewritten if it carries an inline audio data URI.
    def export(self, spans):
        return self._wrapped.export([self._rewrite(s) for s in spans])

    def shutdown(self):
        self._wrapped.shutdown()

    def force_flush(self, timeout_millis=30_000):
        return self._wrapped.force_flush(timeout_millis)

    def _rewrite(self, span):
        attrs = dict(span.attributes or {})
        changed = False
        for key in AUDIO_URL_KEYS:
            val = attrs.get(key)
            if isinstance(val, str) and val.startswith("data:audio/"):
                attrs[key] = self._upload(val, span, key)
                changed = True
        if not changed:
            return span
        # ReadableSpan.attributes is immutable once the span has ended, so we
        # reconstruct the span with the rewritten attributes (every other field
        # passes through unchanged).
        return ReadableSpan(
            name=span.name,
            context=span.context,
            parent=span.parent,
            resource=span.resource,
            attributes=attrs,
            events=span.events,
            links=span.links,
            kind=span.kind,
            status=span.status,
            start_time=span.start_time,
            end_time=span.end_time,
            instrumentation_scope=span.instrumentation_scope,
        )

    def _upload(self, data_uri, span, key):
        # Split "data:audio/wav;base64,<b64>" into header + b64 payload.
        header, _, b64 = data_uri.partition(",")
        mime = header.split(";")[0].removeprefix("data:") or "audio/wav"
        ext = mime.split("/")[-1] or "wav"
        audio_bytes = base64.b64decode(b64)
        # Key the object by trace + span ID so each clip is uniquely addressable
        # back to the span that produced it.
        trace_id = f"{span.context.trace_id:032x}"
        span_id = f"{span.context.span_id:016x}"
        side = key.split(".")[0]  # "input" or "output"
        ts = datetime.utcfromtimestamp(span.start_time / 1e9).strftime("%Y%m%d_%H%M%S")
        path = f"{ts}_{trace_id}_{span_id}_{side}.{ext}"
        if self._prefix:
            path = f"{self._prefix}/{path}"
        blob = self._bucket.blob(path)
        blob.upload_from_string(audio_bytes, content_type=mime)
        return f"https://storage.cloud.google.com/{self._bucket.name}/{path}"


# Allow the instrumentor to ship the full audio inline — our exporter will strip
# the base64 and replace the URL with a GCS link before the span reaches Arize.
os.environ["OPENINFERENCE_BASE64_AUDIO_MAX_LENGTH"] = "100000000"

# Replicate the OTLP-auth header that arize.otel.register would set for us.
os.environ["OTEL_EXPORTER_OTLP_TRACES_HEADERS"] = (
    f"space_id={os.environ['ARIZE_SPACE_ID']},api_key={os.environ['ARIZE_API_KEY']}"
)

# Build the exporter chain: OpenInference span -> our GCS-rewriting wrapper -> OTLP -> Arize.
resource = Resource(attributes={"model_id": PROJECT_NAME})
otlp_exporter = OTLPSpanExporter(endpoint="https://otlp.arize.com/v1")
gcs_exporter = GCSAudioUploadingExporter(
    wrapped=otlp_exporter,
    bucket_name=GCS_BUCKET,
    path_prefix=GCS_PATH,
)

tracer_provider = TracerProvider(resource=resource)
tracer_provider.add_span_processor(SimpleSpanProcessor(gcs_exporter))
trace_api.set_tracer_provider(tracer_provider)

# Same patch as Variant 1 — emit spans through the (manually configured) provider.
OpenAIAgentsInstrumentor().instrument(tracer_provider=tracer_provider)

## Running a voice session

This is the live mic/speaker loop. It opens an `agents.realtime.RealtimeSession` via the `RealtimeRunner`, pumps mic audio into it, plays back the assistant's audio, and exits cleanly after one full conversational turn — including any tool calls and the assistant's follow-up response that uses the tool result.

A few things worth understanding before running it:

**Three async tasks run concurrently inside the session:**

- `send_mic` — drains the mic queue and forwards chunks to the session via `session.send_audio(...)`
- `handle_events` — consumes the session's event stream (`async for event in session`) and dispatches on type:
  - `RealtimeAudio` → write to the output stream (queued for playback)
  - `RealtimeAudioInterrupted` → user spoke over the assistant; abort and re-arm the output stream
  - `RealtimeAgentStartEvent` / `RealtimeToolStart` → "more is coming" — cancel any pending exit
  - `RealtimeAgentEndEvent` → "a response just ended" — schedule a deferred exit
- `hard_timer` — sets `stop` after `MAX_SESSION_SECONDS` as a safety net

**Why the exit is deferred** rather than fired directly on `RealtimeAgentEndEvent`: the SDK's `RealtimeAgentEndEvent` fires after every `response.done` from the API, which includes the response where the model decides to call a tool — *before* the tool runs and the follow-up response. So we schedule the exit after `EXIT_GRACE_PERIOD`, and cancel it if a `RealtimeToolStart` or a new `RealtimeAgentStartEvent` follows within the window (both indicate the turn isn't really over). For tool-free turns the grace window simply elapses and the session exits.

**Try asking:**

- *"What's the weather in London?"* — exercises a tool call (`get_weather`)
- *"What time is it in Tokyo?"* — exercises the other tool (`get_current_time`)
- *"Tell me a fun fact about Python"* — no tool call; tests the simpler one-response path

Server-side VAD (the Realtime API's default `turn_detection`) detects when you've finished speaking and commits your audio buffer automatically. Interrupt the kernel to stop early.

In [ ]:
SAMPLE_RATE = 24_000             # OpenAI Realtime API speaks 24 kHz PCM16 mono on the wire
CHANNELS = 1
MIC_CHUNK_FRAMES = 1_200         # 50 ms blocks
MAX_SESSION_SECONDS = 60         # hard ceiling — session ends after this no matter what
EXIT_GRACE_PERIOD = 1.5          # seconds after AgentEndEvent before exit fires, unless cancelled


async def run_session():
    loop = asyncio.get_running_loop()

    # mic_queue: PortAudio thread -> async send loop.
    # stop: set when we want every task (mic, events, timer) to wind down.
    mic_queue: asyncio.Queue = asyncio.Queue(maxsize=100)
    stop = asyncio.Event()

    async def hard_timer():
        """Safety net — guarantees the session exits even if no events fire."""
        await asyncio.sleep(MAX_SESSION_SECONDS)
        stop.set()

    runner = RealtimeRunner(agent)
    # `runner.run()` returns an async context manager that opens the Realtime
    # WebSocket session; `await` it once, then enter with `async with`.
    async with await runner.run() as session:
        print("Connected. Speak now — session will end once the assistant finishes its turn.\n")

        async def send_mic():
            """Drain mic_queue and forward chunks to the session.

            `wait_for(..., timeout=0.1)` lets us check `stop` ~10x/s instead of
            blocking forever on an empty queue.
            """
            while not stop.is_set():
                try:
                    chunk = await asyncio.wait_for(mic_queue.get(), timeout=0.1)
                    await session.send_audio(chunk)
                except asyncio.TimeoutError:
                    continue

        timer_task = asyncio.create_task(hard_timer())
        send_task = asyncio.create_task(send_mic())
        events_task = None  # created below once the audio streams are open

        try:
            # Open mic + speaker streams. They start automatically on `with` entry
            # and are closed on exit. The output stream uses no callback — we'll
            # write to it synchronously from handle_events.
            with sd.InputStream(
                samplerate=SAMPLE_RATE,
                channels=CHANNELS,
                dtype="int16",
                blocksize=MIC_CHUNK_FRAMES,
                callback=make_mic_callback(mic_queue, loop),
            ), sd.OutputStream(
                samplerate=SAMPLE_RATE,
                channels=CHANNELS,
                dtype="int16",
                blocksize=MIC_CHUNK_FRAMES,
            ) as out_stream:

                async def handle_events():
                    """Consume the session's event stream and drive playback + exit logic."""
                    exit_task = None  # the pending "session done" timer, if any

                    async def maybe_exit():
                        """Wait the grace period, then drain playback and stop the session."""
                        try:
                            await asyncio.sleep(EXIT_GRACE_PERIOD)
                            # `out_stream.stop()` is blocking — it returns only after
                            # PortAudio has finished playing every queued sample.
                            await loop.run_in_executor(None, out_stream.stop)
                            stop.set()
                        except asyncio.CancelledError:
                            pass

                    def cancel_pending_exit():
                        nonlocal exit_task
                        if exit_task is not None and not exit_task.done():
                            exit_task.cancel()
                        exit_task = None

                    async for event in session:
                        if stop.is_set():
                            break
                        if isinstance(event, RealtimeAudio):
                            # Assistant audio chunk — write to the speaker. run_in_executor
                            # keeps the blocking PortAudio write off the event loop.
                            arr = np.frombuffer(event.audio.data, dtype=np.int16)
                            await loop.run_in_executor(None, out_stream.write, arr)
                        elif isinstance(event, RealtimeAudioInterrupted):
                            # User talked over the assistant — drop queued audio and
                            # re-arm the stream for the next reply.
                            cancel_pending_exit()
                            await loop.run_in_executor(None, out_stream.abort)
                            await loop.run_in_executor(None, out_stream.start)
                            print("[interrupted]")
                        elif isinstance(event, (RealtimeAgentStartEvent, RealtimeToolStart)):
                            # New response or tool call — there's more coming.
                            cancel_pending_exit()
                        elif isinstance(event, RealtimeAgentEndEvent):
                            # A response just ended. Maybe final, maybe before a tool call.
                            # Schedule exit; a follow-up AgentStart or ToolStart will cancel it.
                            cancel_pending_exit()
                            exit_task = asyncio.create_task(maybe_exit())

                events_task = asyncio.create_task(handle_events())
                await stop.wait()
        except KeyboardInterrupt:
            print("\nInterrupted.")
            stop.set()
        finally:
            # Cancel any task that was actually created and wait for them to unwind.
            pending = [t for t in (timer_task, send_task, events_task) if t is not None]
            for t in pending:
                t.cancel()
            await asyncio.gather(*pending, return_exceptions=True)

    # Ensure every batched span reaches Arize before the cell returns.
    tracer_provider.force_flush()
    print("Session ended. Traces flushed to Arize.")


await run_session()

## See your traces in Arize

Head to your Arize AX project (`openai-realtime-voice`, unless you changed `PROJECT_NAME`) to see the traces. Each conversation turn appears as a `conversation.turn` span (kind: `AUDIO`) containing `user`, `assistant`, and any `<tool_name>` children.

![An audio trace](https://storage.googleapis.com/arize-phoenix-assets/assets/images/arize-docs-images/cookbooks/voice/trace-audio.png)

Things to look at in the trace view:

- **Audio playback** — click the play button on the `user` and `assistant` spans. The audio is served from whichever URL the span carries: the inline `data:audio/wav;base64,...` URI (Variant 1) or the `https://storage.cloud.google.com/...` URL (Variant 2).
- **Transcripts** — the `input.audio.transcript` and `output.audio.transcript` attributes show what the model heard and what it said. Useful for debugging mishearings.
- **Tool calls** — child `TOOL` spans under the `assistant` span show the function name, arguments JSON, and the value the function returned.
- **Latency** — `time_to_first_token_ms` on the `assistant` span gives the user-perceived "how fast did the agent start talking" latency. Look for outliers across turns.
- **Audio token cost** — `llm.token_count.prompt_details.audio` and `llm.token_count.completion_details.audio` break out audio tokens separately from text tokens.

### Audio redaction

The OpenInference instrumentor recognises three environment variables for controlling what audio data ends up on spans. Set them **before** calling `instrument(...)`:

- `OPENINFERENCE_HIDE_INPUT_AUDIO=true` — drop `input.audio.*` from `USER` spans
- `OPENINFERENCE_HIDE_OUTPUT_AUDIO=true` — drop `output.audio.*` from `LLM` spans
- `OPENINFERENCE_BASE64_AUDIO_MAX_LENGTH=<n>` — cap the inline base64 payload length (default `32000`)

The general OpenInference `TraceConfig(hide_inputs=True)` and `TraceConfig(hide_outputs=True)` settings also cascade to the corresponding audio attributes, so a single redaction config covers both text and audio.

## Read more

- [OpenInference OpenAI Agents instrumentor](https://github.com/Arize-ai/openinference/tree/main/python/instrumentation/openinference-instrumentation-openai-agents) — source, changelog, and the canonical `realtime_with_tools.py` example this notebook is adapted from
- [OpenAI Agents SDK realtime docs](https://openai.github.io/openai-agents-python/realtime/quickstart/) — `RealtimeAgent`, `RealtimeRunner`, and the full event reference
- [OpenAI Realtime API reference](https://platform.openai.com/docs/guides/realtime) — the underlying WebSocket protocol the SDK speaks
- [OpenInference semantic conventions](https://github.com/Arize-ai/openinference/tree/main/spec) — the formal definitions for span kinds and audio attributes
- [Arize AX docs — Tracing and evaluating voice applications](https://docs.arize.com/ax/cookbooks/evaluation/tracing-and-evaluating-audio) — Arize's cookbook on instrumenting and evaluating audio applications